In [44]:
import time
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler


In [45]:
df = pd.read_csv('../data/gym_churn_us.csv')
df['caida_frecuencia'] = df['Avg_class_frequency_total'] - df['Avg_class_frequency_current_month']
X = df.drop(columns=['Churn', 'Avg_class_frequency_total', 'Phone', 'gender'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [46]:
columnas_campana = ['Age','Avg_class_frequency_current_month', 'caida_frecuencia']
columnas_sesgadas_o_fijas = ['Lifetime', 'Avg_additional_charges_total', 'Contract_period', 'Month_to_end_contract']

In [47]:
preprocesador = ColumnTransformer(
    transformers=[
        ('estandarizacion', StandardScaler(), columnas_campana),
        ('normalizacion', MinMaxScaler(), columnas_sesgadas_o_fijas)
    ],
    remainder='passthrough' 
)

In [48]:
X_train_escalado = preprocesador.fit_transform(X_train)
X_test_escalado = preprocesador.transform(X_test)

In [49]:
ros = RandomOverSampler()
X_ros, y_ros = ros.fit_resample(X_train_escalado, y_train)

rus = RandomUnderSampler()
X_rus, y_rus = rus.fit_resample(X_train_escalado, y_train)

smote = SMOTE()
X_smote, y_smote = smote.fit_resample(X_train_escalado, y_train)

In [50]:
X_train = X_smote
y_train = y_smote

In [51]:
model = RandomForestClassifier(n_estimators=100, random_state=42)

In [52]:
y_train_dum = pd.get_dummies(y_train)
y_test_dum = pd.get_dummies(y_test)

In [53]:
class RedPrediccionChurn(nn.Module):
    def __init__(self, input_dim):
        super(RedPrediccionChurn, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 10),
            nn.ReLU(),
            nn.Dropout(0.1), 
            
            nn.Linear(10, 5),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(5, 2),
            
        )
        
    def forward(self, x):
        return self.network(x)


num_caracteristicas = 11 
modelo = RedPrediccionChurn(input_dim=num_caracteristicas)

optimizador = optim.Adam(modelo.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [55]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_val_tensor = torch.tensor(X_test_escalado, dtype=torch.float32)
y_val_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)